# Napari Single-Session QC
This notebook focuses on Napari-based visual QC for one baseline acquisition.

It overlays raw and normalized frames with ROI, center pixel, and patch annotations.

Author: Brynn Harris-Shanks


## Set-up
### Imports


In [1]:

from pathlib import Path
import sys
import yaml
import os

# Locate repo root
repo_root = next(
    p for p in (Path.cwd(), *Path.cwd().parents)
    if (p / "config" / "config.yml").exists()
)

# Add code directory to import path
code_dir = repo_root / "code"
if str(code_dir) not in sys.path:
    sys.path.insert(0, str(code_dir))

# Load config
config_path = repo_root / "config" / "config.yml"
config = yaml.safe_load(config_path.read_text(encoding="utf-8"))

In [2]:
from IPython.display import display
from IPython import get_ipython
import matplotlib
import matplotlib.pyplot as plt
from matplotlib import animation
from matplotlib.colors import TwoSlopeNorm
from matplotlib.patches import Rectangle
import numpy as np
from utils import helper_functions as hf

In [3]:
# Inline backend
USE_WIDGET = False
ipy = get_ipython()
backend_mode = "inline"
if ipy is not None:
    try:
        ipy.run_line_magic("matplotlib", "inline")
    except Exception:
        pass
print(f"Matplotlib backend: {matplotlib.get_backend()} | mode={backend_mode}")

# Core paths
deriv_root = repo_root / config["paths"]["preprocessing"]
subjects = config["subjects"]["all"]
sample_subject = config["subjects"]["default"]
data_mode = "zscore"
sample_session_idx = 2
sample_frame_idx = 0

# Napari config (optional single-session QC)
ENABLE_NAPARI = True
NAPARI_VIEWER_MODE = "inline"  # "inline" or "qt"
NAPARI_PATCH_RADIUS = 1
NAPARI_CENTER_PIXEL = (49, 21)
NAPARI_AUTO_ROI_PERCENTILE = 35.0
NAPARI_AUTO_ROI_MIN_PIXELS = 500



Matplotlib backend: inline | mode=inline


### Load Baseline Data

Iterates over all subjects, loads all baseline sessions from either:
- baseline_only if data_mode == "raw", or
- baseline_only_normalized/{data_mode} (zscore or mean_divide)


Prints dataset summaries (session counts, frame counts, shapes, value ranges).

In [4]:
for subject in subjects:
    data_output_dir = deriv_root / subject  # subject-specific output folder
    data_output_dir.mkdir(parents=True, exist_ok=True)
    if data_mode == "raw":
        baseline_output_dir = os.path.join(data_output_dir, "baseline_only")
    else:
        baseline_output_dir = os.path.join(
            data_output_dir, f"baseline_only_standardized/{data_mode}"
        )
    # Load all baseline sessions
    baseline_sessions = hf.load_saved_baseline_sessions(baseline_output_dir)

    # Print summary
    if len(baseline_sessions) > 0:
        print(f"\nBaseline Data Summary:{subject} ({data_mode})")
        print(f"  Total sessions: {len(baseline_sessions)}")
        total_frames = sum(s["frames"].shape[0] for s in baseline_sessions)
        print(f"  Total baseline frames: {total_frames:,}")

        spatial_shapes = [s["frames"].shape[1:] for s in baseline_sessions]
        unique_shapes = set(spatial_shapes)
        print(f"  Spatial dimensions: {unique_shapes}")

        # Show frame count distribution
        frame_counts = [s["frames"].shape[0] for s in baseline_sessions]
        print(
            f"  Frames per session: min={min(frame_counts)}, max={max(frame_counts)}, "
            f"mean={np.mean(frame_counts):.0f}, std={np.std(frame_counts):.0f}"
        )

        # first session
        if len(baseline_sessions) > 0:
            first_session = baseline_sessions[0]
            print(f"\n  First session ({first_session['session_id']}):")
            print(f"    Frames: {first_session['frames'].shape[0]}")
            print(f"    Shape: {first_session['frames'].shape}")
            print(f"    Dtype: {first_session['frames'].dtype}")
            print(
                f"Value range: [{first_session['frames'].min():.2f}, {first_session['frames'].max():.2f}]"
            )


Baseline Data Summary:secundo (zscore)
  Total sessions: 34
  Total baseline frames: 14,043
  Spatial dimensions: {(107, 128), (132, 128), (101, 128), (112, 128), (96, 128), (91, 128), (122, 128)}
  Frames per session: min=1, max=906, mean=413, std=202

  First session (Se01072020):
    Frames: 526
    Shape: (526, 101, 128)
    Dtype: float32
Value range: [-4.49, 10.65]

Baseline Data Summary:gus (zscore)
  Total sessions: 13
  Total baseline frames: 5,838
  Spatial dimensions: {(132, 128), (101, 128), (112, 128), (96, 128), (81, 128), (122, 128)}
  Frames per session: min=258, max=549, mean=449, std=123

  First session (Gu03092020):
    Frames: 539
    Shape: (539, 112, 128)
    Dtype: float32
Value range: [-4.97, 11.45]


---

### Load Normalized Baseline Frames

First loads raw sessions for sample_subject to pick a session_id. 

Then loads normalized frames for that one session with:hf.load_saved_session_frames(..., "mean_divide")


Selects one frame: frame = mean_frames[sample_frame_idx]

In [5]:
# summary loop
for subject in subjects:
    data_output_dir = deriv_root / subject
    mode_dir = (
        data_output_dir / "baseline_only"
        if data_mode == "raw"
        else data_output_dir / "baseline_only_standardized" / data_mode
    )
    subject_sessions = hf.load_saved_baseline_sessions(str(mode_dir))  # <- not baseline_sessions

# sample session load (always from sample_subject)
sample_raw_dir = deriv_root / sample_subject / "baseline_only"
sample_sessions = hf.load_saved_baseline_sessions(str(sample_raw_dir))

if not (0 <= sample_session_idx < len(sample_sessions)):
    raise ValueError(f"sample_session_idx out of range [0, {len(sample_sessions)-1}]")

sample_session = sample_sessions[sample_session_idx]
mean_frames = hf.load_saved_session_frames(
    deriv_root,
    sample_subject,
    sample_session["session_id"],
    data_mode,
)

if not (0 <= sample_frame_idx < mean_frames.shape[0]):
    raise ValueError(
        f"sample_frame_idx={sample_frame_idx} out of range [0, {mean_frames.shape[0] - 1}]"
    )

frame = mean_frames[sample_frame_idx]


## Napari Single-Session QC (optional)

Overlay raw + selected normalized frames with ROI, patch, and center annotations.


In [6]:
from utils import napari_viz as nv

if not ENABLE_NAPARI:
    print("Napari QC disabled (ENABLE_NAPARI=False)")
else:
    session_id = sample_session["session_id"]
    stdz_mode = data_mode

    raw_frames = None
    stdz_frames = None

    try:
        raw_frames = hf.load_saved_session_frames(
            deriv_root,
            sample_subject,
            session_id,
            "raw",
        )
        raw_frames = hf.squeeze_frames(raw_frames).astype(np.float32, copy=False)
    except Exception as e:
        print(f"Could not load raw frames for Napari QC: {e}")

    if raw_frames is None:
        print("Skipping Napari QC because raw frames are unavailable.")
    else:
        if stdz_mode == "raw":
            stdz_frames = raw_frames
        else:
            try:
                stdz_frames = hf.load_saved_session_frames(
                    deriv_root,
                    sample_subject,
                    session_id,
                    stdz_mode,
                )
                stdz_frames = hf.squeeze_frames(stdz_frames).astype(np.float32, copy=False)
            except Exception as e:
                print(
                    f"Could not load standardized frames for mode '{stdz_mode}' "
                    f"(session {session_id}): {e}"
                )

        if stdz_frames is None:
            print("Skipping Napari QC because standardized frames are unavailable.")
        elif raw_frames.shape[1:] != stdz_frames.shape[1:]:
            print(
                "Skipping Napari QC because spatial shapes differ between raw and standardized: "
                f"{raw_frames.shape[1:]} vs {stdz_frames.shape[1:]}"
            )
        else:
            if raw_frames.shape[0] != stdz_frames.shape[0]:
                t_common = min(raw_frames.shape[0], stdz_frames.shape[0])
                print(
                    "Frame count mismatch between raw and standardized; trimming to "
                    f"T={t_common} (raw={raw_frames.shape[0]}, norm={stdz_frames.shape[0]})."
                )
                raw_frames = raw_frames[:t_common]
                stdz_frames = stdz_frames[:t_common]

            t_total, h, w = raw_frames.shape
            start_idx = int(np.clip(sample_frame_idx, 0, t_total - 1))

            roi_path = Path(f"roi_{sample_session_idx}.npy")
            roi_mask = None
            if roi_path.exists():
                try:
                    loaded_roi = np.load(roi_path)
                    if loaded_roi.shape == (h, w):
                        roi_mask = loaded_roi.astype(bool)
                        print(f"Loaded ROI mask: {roi_path}")
                    else:
                        print(
                            f"ROI mask shape mismatch ({loaded_roi.shape} != {(h, w)}); "
                            "using in-memory auto ROI instead."
                        )
                except Exception as e:
                    print(f"Failed to load ROI mask from {roi_path}: {e}")

            if roi_mask is None:
                roi_mask = nv.compute_auto_roi_mask(
                    raw_frames,
                    percentile=NAPARI_AUTO_ROI_PERCENTILE,
                    min_pixels=NAPARI_AUTO_ROI_MIN_PIXELS,
                )
                print(
                    "Computed in-memory auto ROI mask "
                    f"| pixels={int(roi_mask.sum())} ({100.0 * float(roi_mask.mean()):.1f}%)"
                )

            inline_ready, missing = nv.check_napari_inline_ready()
            if NAPARI_VIEWER_MODE == "inline" and not inline_ready:
                miss = ", ".join(missing)
                print(f"Napari inline dependencies missing: {miss}")
                print('Install with: pip install "napari[all]" magicgui ipywidgets')
            else:
                center_yx = (
                    int(np.clip(NAPARI_CENTER_PIXEL[0], 0, h - 1)),
                    int(np.clip(NAPARI_CENTER_PIXEL[1], 0, w - 1)),
                )

                try:
                    napari_viewer = nv.launch_single_session_qc_viewer(
                        raw_frames=raw_frames,
                        stdz_frames=stdz_frames,
                        mode_name=stdz_mode,
                        session_label=f"{sample_subject} | session {session_id}",
                        roi_mask=roi_mask,
                        center_yx=center_yx,
                        patch_radius=NAPARI_PATCH_RADIUS,
                        start_frame=start_idx,
                        viewer_mode=NAPARI_VIEWER_MODE,
                    )

                    if NAPARI_VIEWER_MODE == "inline":
                        display(napari_viewer)
                    else:
                        print("Napari viewer opened in Qt mode.")
                except Exception as e:
                    print(f"Napari viewer failed to launch: {e}")
                    if NAPARI_VIEWER_MODE == "inline":
                        print('Try desktop fallback: set NAPARI_VIEWER_MODE = "qt" and rerun this cell.')



Computed in-memory auto ROI mask | pixels=9823 (62.9%)
Napari viewer failed to launch: Napari inline widget backend is unavailable in this kernel. Install/enable `jupyter_rfb` + `ipywidgets`, or use `NAPARI_VIEWER_MODE = "qt"`.
Try desktop fallback: set NAPARI_VIEWER_MODE = "qt" and rerun this cell.
